In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,r2_score,classification_report,confusion_matrix,mean_squared_error
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder,PolynomialFeatures
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier


In [48]:
data = pd.read_csv("adult.csv")
data.head()


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [49]:
data.duplicated().sum()
data.drop_duplicates(inplace=True)


In [50]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48790 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48790 non-null  int64 
 1   workclass        48790 non-null  object
 2   fnlwgt           48790 non-null  int64 
 3   education        48790 non-null  object
 4   educational-num  48790 non-null  int64 
 5   marital-status   48790 non-null  object
 6   occupation       48790 non-null  object
 7   relationship     48790 non-null  object
 8   race             48790 non-null  object
 9   gender           48790 non-null  object
 10  capital-gain     48790 non-null  int64 
 11  capital-loss     48790 non-null  int64 
 12  hours-per-week   48790 non-null  int64 
 13  native-country   48790 non-null  object
 14  income           48790 non-null  object
dtypes: int64(6), object(9)
memory usage: 6.0+ MB


In [51]:
data.workclass.unique()

array(['Private', 'Local-gov', '?', 'Self-emp-not-inc', 'Federal-gov',
       'State-gov', 'Self-emp-inc', 'Without-pay', 'Never-worked'],
      dtype=object)

In [52]:
Most_repeated_Value = data.workclass.mode()[0]

In [53]:
data.workclass = data.workclass.str.replace("?",Most_repeated_Value)

In [54]:
data.education.unique()

array(['11th', 'HS-grad', 'Assoc-acdm', 'Some-college', '10th',
       'Prof-school', '7th-8th', 'Bachelors', 'Masters', 'Doctorate',
       '5th-6th', 'Assoc-voc', '9th', '12th', '1st-4th', 'Preschool'],
      dtype=object)

In [55]:
data.education = data.education.str.strip().str.replace('-',' ')

In [56]:
education_change = str(np.random.choice(['7th','8th','1st','4th','5th','6th']))

In [57]:
data.education = data.education.replace(['7th-8th','5th-6th','1st-4th','-'],[education_change,education_change,education_change,' '])

In [58]:
data.info()


<class 'pandas.core.frame.DataFrame'>
Index: 48790 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48790 non-null  int64 
 1   workclass        48790 non-null  object
 2   fnlwgt           48790 non-null  int64 
 3   education        48790 non-null  object
 4   educational-num  48790 non-null  int64 
 5   marital-status   48790 non-null  object
 6   occupation       48790 non-null  object
 7   relationship     48790 non-null  object
 8   race             48790 non-null  object
 9   gender           48790 non-null  object
 10  capital-gain     48790 non-null  int64 
 11  capital-loss     48790 non-null  int64 
 12  hours-per-week   48790 non-null  int64 
 13  native-country   48790 non-null  object
 14  income           48790 non-null  object
dtypes: int64(6), object(9)
memory usage: 6.0+ MB


In [59]:
data['marital-status'] = data['marital-status'].str.strip().str.replace('-',' ')

In [60]:
data.occupation.unique()

array(['Machine-op-inspct', 'Farming-fishing', 'Protective-serv', '?',
       'Other-service', 'Prof-specialty', 'Craft-repair', 'Adm-clerical',
       'Exec-managerial', 'Tech-support', 'Sales', 'Priv-house-serv',
       'Transport-moving', 'Handlers-cleaners', 'Armed-Forces'],
      dtype=object)

In [61]:
data.occupation = data.occupation.str.strip().str.replace('-',' ').str.replace("?",data.occupation.mode()[0])

In [62]:
data.occupation.mode()[0]

'Prof specialty'

In [63]:
data.relationship.unique()

array(['Own-child', 'Husband', 'Not-in-family', 'Unmarried', 'Wife',
       'Other-relative'], dtype=object)

In [64]:
data.relationship = data.relationship.str.strip().str.replace("-",' ')

In [65]:
data.race.unique()

array(['Black', 'White', 'Asian-Pac-Islander', 'Other',
       'Amer-Indian-Eskimo'], dtype=object)

In [66]:
data[(data.select_dtypes("object")).replace('-',' ',regex=True).columns] = (data.select_dtypes("object")).replace('-',' ',regex=True)

In [67]:
data

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never married,Machine op inspct,Own child,Black,Male,0,0,40,United States,<=50K
1,38,Private,89814,HS grad,9,Married civ spouse,Farming fishing,Husband,White,Male,0,0,50,United States,<=50K
2,28,Local gov,336951,Assoc acdm,12,Married civ spouse,Protective serv,Husband,White,Male,0,0,40,United States,>50K
3,44,Private,160323,Some college,10,Married civ spouse,Machine op inspct,Husband,Black,Male,7688,0,40,United States,>50K
4,18,Private,103497,Some college,10,Never married,Prof specialty,Own child,White,Female,0,0,30,United States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,257302,Assoc acdm,12,Married civ spouse,Tech support,Wife,White,Female,0,0,38,United States,<=50K
48838,40,Private,154374,HS grad,9,Married civ spouse,Machine op inspct,Husband,White,Male,0,0,40,United States,>50K
48839,58,Private,151910,HS grad,9,Widowed,Adm clerical,Unmarried,White,Female,0,0,40,United States,<=50K
48840,22,Private,201490,HS grad,9,Never married,Adm clerical,Own child,White,Male,0,0,20,United States,<=50K


In [68]:
for i in data.select_dtypes("object").columns:
    uni = data[i].unique()
    print(f'[------{i}-----] :: {uni}')




[------workclass-----] :: ['Private' 'Local gov' 'Self emp not inc' 'Federal gov' 'State gov'
 'Self emp inc' 'Without pay' 'Never worked']
[------education-----] :: ['11th' 'HS grad' 'Assoc acdm' 'Some college' '10th' 'Prof school'
 '7th 8th' 'Bachelors' 'Masters' 'Doctorate' '5th 6th' 'Assoc voc' '9th'
 '12th' '1st 4th' 'Preschool']
[------marital-status-----] :: ['Never married' 'Married civ spouse' 'Widowed' 'Divorced' 'Separated'
 'Married spouse absent' 'Married AF spouse']
[------occupation-----] :: ['Machine op inspct' 'Farming fishing' 'Protective serv' 'Prof specialty'
 'Other service' 'Craft repair' 'Adm clerical' 'Exec managerial'
 'Tech support' 'Sales' 'Priv house serv' 'Transport moving'
 'Handlers cleaners' 'Armed Forces']
[------relationship-----] :: ['Own child' 'Husband' 'Not in family' 'Unmarried' 'Wife' 'Other relative']
[------race-----] :: ['Black' 'White' 'Asian Pac Islander' 'Other' 'Amer Indian Eskimo']
[------gender-----] :: ['Male' 'Female']
[------native-co

In [69]:
data['native-country'] = data['native-country'].replace("?",data['native-country'].mode()[0])

In [70]:
data['native-country'].unique()

array(['United States', 'Peru', 'Guatemala', 'Mexico',
       'Dominican Republic', 'Ireland', 'Germany', 'Philippines',
       'Thailand', 'Haiti', 'El Salvador', 'Puerto Rico', 'Vietnam',
       'South', 'Columbia', 'Japan', 'India', 'Cambodia', 'Poland',
       'Laos', 'England', 'Cuba', 'Taiwan', 'Italy', 'Canada', 'Portugal',
       'China', 'Nicaragua', 'Honduras', 'Iran', 'Scotland', 'Jamaica',
       'Ecuador', 'Yugoslavia', 'Hungary', 'Hong', 'Greece',
       'Trinadad&Tobago', 'Outlying US(Guam USVI etc)', 'France',
       'Holand Netherlands'], dtype=object)

In [71]:
data['income'].value_counts()

income
<=50K    37109
>50K     11681
Name: count, dtype: int64

In [72]:
data.describe()

,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,48790.000000,4.879000e+04,48790.000000,48790.000000,48790.000000,48790.000000
mean,38.652798,1.896690e+05,10.078807,1080.217688,87.595573,40.425886
std,13.708493,1.056172e+05,2.570046,7455.905921,403.209129,12.392729
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175550e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781385e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376062e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


In [73]:
from sklearn.preprocessing import LabelEncoder

encode = {}

for col in data.select_dtypes("object").columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encode[col] = le

In [74]:
x = data.drop('income',axis=1)

y = data.income

In [75]:
smote = SMOTE(sampling_strategy="auto",random_state=42)
X_smote,y_smote = smote.fit_resample(x,y)
data = pd.concat([X_smote,y_smote],axis=1)

In [76]:
for col in encode.keys():
    data[col] = encode[col].inverse_transform(data[col])

In [77]:
data.income.value_counts()

income
<=50K    37109
>50K     37109
Name: count, dtype: int64

In [78]:
x = data.drop('income',axis=1)

y = data.income

In [79]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.20,random_state=100)

In [80]:
num_column = X_train.select_dtypes("number")
Obj_column = X_train.select_dtypes("object")



In [81]:
def create_model(X_train,y_train,model):
    number_data = Pipeline(
        [
    ('scale',StandardScaler()),
            ('filling',SimpleImputer(strategy="median"))
        ]
    )
    object_data = Pipeline(
        [
            ("encode",OneHotEncoder(drop="first",sparse_output=False,handle_unknown="ignore")),
            ('filling',SimpleImputer(strategy="most_frequent"))
        ]
    )
    Preprocessing = ColumnTransformer(
        transformers=[
            ('num',number_data,make_column_selector(dtype_include=np.number)),
            ('Obj',object_data,make_column_selector(dtype_include="object"))
        ]
    )
    model_Pipeline = Pipeline(
        [
            ('Preprocessor',Preprocessing),
            ('model',model),
            
        ]
    )
    
    model_Pipeline.fit(X_train,y_train)
    return model_Pipeline


In [82]:
def Model_Predict(X_test,y_test,Model):
    y_pre = Model.predict(X_test)
    print(f'Accuracy Score :       {accuracy_score(y_test,y_pre)}')
    print(f'Confusion matrix :     {confusion_matrix(y_test,y_pre)}')
    print(f"Classification matrix : {classification_report(y_test,y_pre)}")
    

In [83]:
SVC = create_model(X_train,y_train,SVC(kernel="poly"))

In [84]:
Model_Predict(X_test,y_test,SVC)

Accuracy Score :       0.887429264349232
Confusion matrix :     [[6583  816]
 [ 855 6590]]
Classification matrix :               precision    recall  f1-score   support

       <=50K       0.89      0.89      0.89      7399
        >50K       0.89      0.89      0.89      7445

    accuracy                           0.89     14844
   macro avg       0.89      0.89      0.89     14844
weighted avg       0.89      0.89      0.89     14844



In [85]:
Logistic_Regression = create_model(X_train,y_train,LogisticRegression())
Model_Predict(X_test,y_test,Logistic_Regression)


Accuracy Score :       0.8608865534896254
Confusion matrix :     [[6354 1045]
 [1020 6425]]
Classification matrix :               precision    recall  f1-score   support

       <=50K       0.86      0.86      0.86      7399
        >50K       0.86      0.86      0.86      7445

    accuracy                           0.86     14844
   macro avg       0.86      0.86      0.86     14844
weighted avg       0.86      0.86      0.86     14844



In [86]:
Decisiontree = create_model(X_train,y_train,DecisionTreeClassifier(random_state=42))
Model_fit(X_test,y_test,Decisiontree)


Accuracy Score :  0.8592023713284829
Confusion matrix : [[6328 1071]
 [1019 6426]]
Classification matrix :               precision    recall  f1-score   support

       <=50K       0.86      0.86      0.86      7399
        >50K       0.86      0.86      0.86      7445

    accuracy                           0.86     14844
   macro avg       0.86      0.86      0.86     14844
weighted avg       0.86      0.86      0.86     14844



In [96]:
KNN = create_model(X_train,y_train,KNeighborsClassifier(n_neighbors=3))
Model_fit(X_test,y_test,KNN)


Accuracy Score :  0.8552277014281865
Confusion matrix : [[6366 1033]
 [1116 6329]]
Classification matrix :               precision    recall  f1-score   support

       <=50K       0.85      0.86      0.86      7399
        >50K       0.86      0.85      0.85      7445

    accuracy                           0.86     14844
   macro avg       0.86      0.86      0.86     14844
weighted avg       0.86      0.86      0.86     14844



In [87]:
import pickle
with open("SVC_model.pkl", "wb") as file:
    pickle.dump(SVC, file)

In [90]:
with open("Logistic_model.pkl", "wb") as file:
    pickle.dump(Logistic_Regression ,file)

In [91]:
with open("Decision_tree_model.pkl", "wb") as file:
    pickle.dump(Decisiontree, file)

In [97]:
with open("KNN_model.pkl", "wb") as file:
    pickle.dump(KNN, file)